# Agents: Refine a Design Rationale with Propose/Critic

Start with a reusable LLM pattern before building workflow machinery. This notebook
uses `ProposeCriticPattern` with a small model served by Ollama. The pattern owns the
proposal, structured critique, revision loop, and result contract; the notebook only
configures the model and the design task.

## Setup

1. [Install Ollama](https://ollama.com/download). Start the Ollama app, or keep the
   server running in Terminal A:

   ```bash
   ollama serve
   ```

2. In Terminal B, install the package, fetch the tutorial model, and verify it is listed:

   ```bash
   python -m pip install design-research-agents==0.7.0
   ollama pull qwen3:8b
   ollama list
   ```

Open the notebook in VS Code, select that Python environment, confirm `qwen3:8b` appears
in `ollama list`, and then choose **Run All**. Model wording can vary; the stored output
below is one captured run.

## Step 1: Configure the local model and design task

In [1]:
import design_research_agents as agents

MODEL = "qwen3:8b"
TASK = (
    "Write exactly three one-sentence bullets supporting modular connectors in a "
    "field-serviceable environmental sensor: maintenance, reliability, then one "
    "tradeoff. Do not invent quantitative evidence. No introduction or conclusion."
)
print("Model:", MODEL)
print("Task:", TASK)

Model: qwen3:8b
Task: Write exactly three one-sentence bullets supporting modular connectors in a field-serviceable environmental sensor: maintenance, reliability, then one tradeoff. Do not invent quantitative evidence. No introduction or conclusion.


## Step 2: Run the existing pattern

Custom system prompts define review criteria, while the packaged pattern handles both
model calls and decides whether another revision is needed.

In [2]:
with agents.OllamaLLMClient(
    default_model=MODEL,
    manage_server=False,
    request_timeout_seconds=120,
    max_retries=0,
) as llm_client:
    pattern = agents.ProposeCriticPattern(
        llm_client=llm_client,
        max_iterations=2,
        proposer_system_prompt=(
            "You are a concise engineering proposer. Follow the requested format "
            "exactly. Make only qualitative claims supported by the task; do not "
            "invent numbers."
        ),
        critic_system_prompt=(
            "You are a strict engineering critic. Approve only when there are "
            "exactly three concise bullets addressing maintenance, reliability, "
            "and one tradeoff, with no invented numbers or unsupported evidence. "
            "Return JSON only with approved, feedback, revision_goals."
        ),
    )
    result: agents.ProposeCriticResult = pattern.run(
        TASK, request_id="tutorial-agents-propose-critic"
    )

print("Success:", result.success)
print("Termination:", result.terminated_reason)

Success: True
Termination: approved


## Step 3: Inspect the proposal returned by the pattern

In [3]:
print(result.proposal)
print("Approved:", result.approved)
print("Iterations:", result.iterations)

- Modular connectors simplify maintenance by enabling quick replacement of individual components without full system disassembly.  
- Modular connectors enhance reliability by isolating component failures and allowing targeted repairs without system-wide disruptions.  
- Modular connectors may introduce compatibility complexities and increased part inventory management requirements compared to monolithic designs.
Approved: True
Iterations: 1


## Step 4: Inspect the structured critique history

In [4]:
for iteration in result.critique_iterations:
    print(f"Iteration {iteration['iteration']}")
    print("Approved:", iteration["approved"])
    print("Feedback:", iteration["feedback"] or "(none)")
    print("Revision goals:", iteration["revision_goals"])

if not result.success or not result.approved:
    raise RuntimeError(
        "The live propose/critic workflow did not finish with a successful, approved result."
    )

Iteration 1
Approved: True
Feedback: (none)
Revision goals: []


## Next steps

Change the model, review criteria, or design task before changing orchestration. Use a
home-built workflow only when the protocol needs behavior that an existing pattern
does not already express. The next tutorial builds that workflow explicitly.